In [ ]:
import numpy as np
import math

class NeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.1, seed=None):
        """
        layer_sizes: list of ints, e.g. [784, 128, 10]
        learning_rate: float
        seed: int or None
        """
        if seed is not None:
            np.random.seed(seed)
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        # Initialize weights and biases
        self.weights = [np.random.randn(n_in, n_out) * np.sqrt(2. / n_in)
                        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]
        self.biases = [np.zeros((1, n_out)) for n_out in layer_sizes[1:]]

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        s = self.sigmoid(x)
        return s * (1 - s)

    def forward(self, X):
        """
        Returns activations and z-values for each layer
        """
        activations = [X]
        zs = []
        a = X
        for w, b in zip(self.weights, self.biases):
            z = np.dot(a, w) + b
            zs.append(z)
            a = self.sigmoid(z)
            activations.append(a)
        return activations, zs

    def backward(self, X, y, activations, zs):
        """
        Returns gradients for weights and biases
        """
        grads_w = [np.zeros_like(w) for w in self.weights]
        grads_b = [np.zeros_like(b) for b in self.biases]
        # Output error
        delta = (activations[-1] - y) * self.sigmoid_derivative(zs[-1])
        grads_w[-1] = np.dot(activations[-2].T, delta)
        grads_b[-1] = np.sum(delta, axis=0, keepdims=True)
        # Backpropagate
        for l in range(2, len(self.layer_sizes)):
            z = zs[-l]
            sp = self.sigmoid_derivative(z)
            delta = np.dot(delta, self.weights[-l+1].T) * sp
            grads_w[-l] = np.dot(activations[-l-1].T, delta)
            grads_b[-l] = np.sum(delta, axis=0, keepdims=True)
        return grads_w, grads_b

    def update_params(self, grads_w, grads_b, m):
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * grads_w[i] / m
            self.biases[i] -= self.learning_rate * grads_b[i] / m

    def fit(self, X, y, epochs=10, batch_size=32, verbose=True):
        n = X.shape[0]
        for epoch in range(epochs):
            # Shuffle data
            indices = np.arange(n)
            np.random.shuffle(indices)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            for start in range(0, n, batch_size):
                end = start + batch_size
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]
                activations, zs = self.forward(X_batch)
                grads_w, grads_b = self.backward(X_batch, y_batch, activations, zs)
                self.update_params(grads_w, grads_b, X_batch.shape[0])
            if verbose:
                loss = self.loss(X, y)
                acc = self.evaluate(X, y)
                print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} - Accuracy: {acc:.4f}")

    def predict(self, X):
        activations, _ = self.forward(X)
        return activations[-1]

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        if y.shape[1] == 1:
            # Binary classification
            y_pred_label = (y_pred > 0.5).astype(int)
            return np.mean(y_pred_label == y)
        else:
            # Multiclass classification
            y_pred_label = np.argmax(y_pred, axis=1)
            y_true_label = np.argmax(y, axis=1)
            return np.mean(y_pred_label == y_true_label)

    def loss(self, X, y):
        y_pred = self.predict(X)
        # Use mean squared error for simplicity
        return np.mean((y_pred - y) ** 2)

# Example usage (for MNIST):
# nn = NeuralNetwork([784, 128, 10], learning_rate=0.1)
# nn.fit(X_train, y_train, epochs=10, batch_size=64)
# acc = nn.evaluate(X_test, y_test)
# print("Test accuracy:", acc) 
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

In [ ]:
# Show example images from data/normal/
img_paths = os.listdir("data/head_annotations")
for f in img_paths[:1]:
    img = Image.open(f"data/head_annotations/{f}")
    plt.imshow(img)
    plt.title(f)
    plt.axis("off")
    plt.show()

In [ ]:
# Load pretrained ResNet-50 model and remove the classifier

weights = ResNet50_Weights.DEFAULT  # or IMAGENET1K_V1
resnet = resnet50(weights=weights)
resnet.fc = nn.Identity()
resnet.eval()

In [ ]:
# Image transform for ResNet input
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])



In [ ]:
def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0)
    with torch.no_grad():
        return resnet(x).squeeze().numpy()


In [ ]:
#extract features
""""
normal_feats = []
normal_files = os.listdir("data/head_annotations")

for f in normal_files:
    path = os.path.join("data/head_annotations", f)
    normal_feats.append(extract_features(path))

normal_feats = np.array(normal_feats)
"""


In [ ]:
#np.save("features/normal_head_features(minarea-1000-hospital-1).npy", normal_feats)

In [ ]:
normal_feats = np.load("features/normal_head_features(minarea-1000-hospital-1).npy")
print(normal_feats)

In [ ]:
# PCA or t-SNE to reduce features
pca = PCA(n_components=2)
reduced = pca.fit_transform(normal_feats)

plt.scatter(reduced[:,0], reduced[:,1], alpha=0.6)
plt.title("PCA of Normal Head Feature Vectors")
plt.show()


In [ ]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(),
            nn.Linear(512, 128)
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 512), nn.ReLU(),
            nn.Linear(512, 2048)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

autoencoder = AE()
opt = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Train/val split
train, val = train_test_split(normal_feats, test_size=0.2)
train_loader = DataLoader(torch.tensor(train).float(), batch_size=16, shuffle=True)
val_loader = DataLoader(torch.tensor(val).float(), batch_size=16)

# Training loss
for epoch in range(30):
    autoencoder.train()
    loss_sum = 0
    for batch in train_loader:
        opt.zero_grad()
        out = autoencoder(batch)
        loss = loss_fn(out, batch)
        loss.backward()
        opt.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1} | Train Loss: {loss_sum/len(train_loader):.4f}")


In [ ]:
# autoencoder
torch.save(autoencoder.state_dict(), 'autoencoder.pth')

In [ ]:
threshold = 0.02  # Adjust as needed

test_files = os.listdir("data/test")
for f in test_files:
    img_path = os.path.join("data/test", f)
    img = Image.open(img_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        feat = resnet(img_tensor).squeeze().numpy()

    input_tensor = torch.tensor(feat).float().unsqueeze(0)
    with torch.no_grad():
        recon = autoencoder(input_tensor)

    error = F.mse_loss(recon, input_tensor).item()
    label = "⚠️ Anomaly" if error > threshold else "✅ Normal"
    print(f"{f} → {label} (error={error:.4f})")


In [ ]:
errors = []

for f in test_files:
    feat = extract_features(f"data/test/{f}")
    input_tensor = torch.tensor(feat).float().unsqueeze(0)
    with torch.no_grad():
        recon = autoencoder(input_tensor)
    err = nn.functional.mse_loss(recon, input_tensor).item()
    errors.append((f, err))

# Sort by descending error
errors.sort(key=lambda x: -x[1])
names, values = zip(*errors)

# Plot top 10
plt.figure(figsize=(8, 5))
plt.barh(names[:10], values[:10], color='tomato')
plt.xlabel("Reconstruction Error")
plt.title("Top 10 Most Abnormal Scans")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()